# 🎨 Paint-Code-RL: Zero-Cost GRPO Generative Art on Kaggle GPU

This notebook dynamically detects the Kaggle GPU runtime, configures headless WebGL via Chromium + Xvfb, loads trained LoRA policies, and runs reinforcement learning training cycles.

### ⚡ Key Capabilities:
1. **Node.js 20 LTS & Headless WebGL**: Clean, modern Chromium rendering under virtual X11 buffer (`xvfb-run`).
2. **Auto-Hardware Detection**: Dynamically sets VRAM limits, precision (bf16/fp16), and batch sizes.
3. **Zero Conflicts**: Neutralizes Kaggle's incompatible `torchao` to prevent PEFT import errors.
4. **Instant Model Showcase**: Automatically downloads and evaluates `pernavjain/paint-code/pyTorch/default`.
5. **GRPO Cyclic Training**: Optimizes p5.js syntax and pixel-space visual richness with live dashboard tracking.
6. **One-Click Export**: Bundles all renders and checkpoints into a downloadable ZIP.

In [ ]:
# Cell 1: Environment & GPU Auto-Probing
import os, sys, platform, torch, psutil

print('=' * 60)
print('   PAINT-CODE-RL: KAGGLE RUNTIME INITIALIZATION')
print('=' * 60)
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  [GPU {i}] {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)')
else:
    print('  [WARN] Running on CPU. For fast GPU acceleration, enable GPU in Settings.')

ram = psutil.virtual_memory()
print(f'RAM: {ram.total / 1e9:.1f} GB Total / {ram.available / 1e9:.1f} GB Available')
print('=' * 60)

In [ ]:
# Cell 2: Install Node.js 20 LTS, Chromium, and Xvfb for Headless WebGL
# NodeSource repository ensures Node 20 LTS instead of old Ubuntu 12.x
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs chromium-browser xvfb > /dev/null 2>&1
!node -v && npm -v

In [ ]:
# Cell 3: Clone Repository and Checkout Active Branch
import os
if not os.path.exists("paint-code-rl"):
    !git clone https://github.com/harshitthek/paint-code-rl.git
%cd paint-code-rl
!git checkout feat/visual-rl-and-cyclic-training || git checkout main
!git pull origin feat/visual-rl-and-cyclic-training || true

In [ ]:
# Cell 4: Install Python & Node.js Dependencies (with torchao neutralization)
# Neutralize Kaggle's incompatible pre-installed torchao 0.10.0 to prevent PEFT crash
!pip uninstall -y torchao > /dev/null 2>&1
!pip install -q trl==0.15.1 transformers==4.49.0 peft datasets accelerate pydantic safetensors Pillow pyyaml psutil requests kagglehub
%cd renderer
!npm install --no-audit --no-fund
%cd ..

In [ ]:
# Cell 5: Launch Headless WebGL Rendering Daemon with Xvfb
import subprocess, time, requests

# Start renderer daemon under xvfb virtual frame buffer
proc = subprocess.Popen(["xvfb-run", "-a", "node", "renderer/server.js"])
time.sleep(4)

# Verify renderer health
for attempt in range(5):
    try:
        health = requests.get("http://127.0.0.1:3000/health", timeout=3).json()
        print("[OK] Renderer daemon is live:", health)
        break
    except Exception:
        time.sleep(2)
else:
    print("[WARN] Renderer daemon did not respond immediately, continuing...")

In [ ]:
# Cell 6: Generate Artwork Using Your Uploaded KaggleHub Checkpoint
# Automatically pulls pernavjain/paint-code/pyTorch/default and renders high-res artworks
!python scripts/generate_and_render.py --kagglehub pernavjain/paint-code/pyTorch/default --output-dir artifacts/renders --temperature 0.4 --max-new-tokens 550

import glob
from IPython.display import display, Image

renders = sorted(glob.glob("artifacts/renders/*.png"))
print(f"Total artworks generated: {len(renders)}")
for r in renders:
    print(f"File: {r}")
    display(Image(filename=r, width=400))

In [ ]:
# Cell 7: Run GRPO Cyclic Training on GPU with Auto Hardware Saturation
import os
os.environ["ENV"] = "kaggle"
os.environ["PYTHONUNBUFFERED"] = "1"

# Run 25 training steps with auto hardware saturation and live dashboard
!python scripts/train_grpo.py --mode train --steps-per-cycle 25 --max-steps 25 --unattended --max --dashboard

In [ ]:
# Cell 8: View Live Dashboard Inline
from IPython.display import display, HTML
if os.path.exists("artifacts/dashboard.html"):
    with open("artifacts/dashboard.html", "r", encoding="utf-8") as f:
        html_code = f.read()
    display(HTML(f'<iframe srcdoc="{html_code.replace(chr(34), "&quot;")}" width="100%" height="600px" frameborder="0"></iframe>'))

In [ ]:
# Cell 9: Package All Outputs into Downloadable ZIP
# After running this, download 'paint_rl_artifacts.zip' from Kaggle's right-hand Output panel!
!python scripts/package_artifacts.py --output-dir /kaggle/working
print("\n[DONE] Check the right-hand panel under 'Output' to download your ZIP file!")